### Minimizing QM of MEOH

In [1]:
import numpy as np
from scipy.optimize import minimize
from pyscf import scf, qmmm, gto
from pyscf.tools import cubegen
from pyscf.geomopt.geometric_solver import optimize
import MDAnalysis as mda
import sys; sys.path.append('/home/chemistry/QMMM/')
import visualize_QMMM as vQ
import SCF_helpers as Sh
from importlib import reload  # Python 3.4+
import pickle
import os
import py3Dmol

In [2]:
# ============================================================
# Constants
# ============================================================

HARTREE_TO_KJMOL = 2625.49962

In [3]:
picklefile = "MMP_QM.pkl"
cube_folder = "cubes_optimized_MEMP"

In [4]:
# Extract atom positions from System0.gro
qm_atoms, mm_coords, mm_charges, mm_names, mm_residues = Sh.read_gro(
    "../System0.gro",
    qm_residues=("MMP")
    # mm_residues=("NA", "SOL"),
)

In [5]:
# Creates a PySCF Mole object describing the quantum-mechanical system that PySCF will calculate
mol = gto.M(
    atom=qm_atoms,
    basis="sto-3g",
    charge=0,
    spin=0,
    unit="Angstrom",
    verbose=4
)

RuntimeError: Electron number 57 and spin 0 are not consistent
Note mol.spin = 2S = Nalpha - Nbeta, not 2S+1

In [ ]:
# SCF calculation
mf = scf.RHF(mol)

# Geometry optimization
mol_opt = optimize(mf)

print("\nOptimized geometry:")
print(mol_opt.atom_coords(unit="Angstrom"))

In [ ]:
print("\nOptimized geometry:")
print(mol_opt.atom_coords(unit="Angstrom"))


# ============================================================
# SCF calculation at optimized geometry
# ============================================================

mf_opt = scf.RHF(mol_opt)
energy = mf_opt.kernel()

print("\nOptimized SCF energy:")
print(energy)

print("\nNumber of MOs:")
print(mf_opt.mo_coeff.shape[1])

mo_coeffs = mf_opt.mo_coeff

In [ ]:
# ============================================================
# Generate orbital cube files
# ============================================================

os.makedirs(cube_folder, exist_ok=True)

resolution = 0.3

norbs = mf_opt.mo_coeff.shape[1]
n_occ = sum(mf_opt.mo_occ > 0)

orbital_filenames = []

for iorb in range(norbs):

    filename = f"{cube_folder}/orbital_{iorb}.cube"

    cubegen.orbital(
        mol_opt,
        filename,
        mf_opt.mo_coeff[:, iorb],
        resolution=resolution
    )

    orbital_filenames.append(filename)

print(f"\nGenerated {norbs} orbital cube files.")
print(f"Directory: {cube_folder}")

In [ ]:
qm_data = {
    "mol_opt": mol_opt,
    "mo_coeff": mf_opt.mo_coeff,
    "mo_energy": mf_opt.mo_energy,
    "mo_occ": mf_opt.mo_occ,
    "e_tot": mf_opt.e_tot,
    "n_occ": int(sum(mf_opt.mo_occ > 0)),
    "cube_folder": cube_folder,
    "orbital_filenames": orbital_filenames,
}

with open(picklefile, "wb") as f:
    pickle.dump(qm_data, f)

print(picklefile)